In [1]:
# import libraries
import pandas as pd
import random
import re
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import sys
from pathlib import Path

base_path = Path.cwd() / "../"
sys.path.append(str(base_path.resolve()))
from utils.data_augmentation import find_entity_span, augmentation_entity, augmentation_non_entity

In [ ]:
# load the data and shuffle
with open("../01_data/annotations_reduced.json", "r") as f:
    data = json.load(f)
data = random.sample(data)

In [2]:
# load the data
with open("../01_data/annotations_reduced.json", "r") as f:
    data = json.load(f)

# load the tokenizer and model and move to device
checkpoint = "HuggingFaceTB/SmolLM-1.7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model = AutoModelForCausalLM.from_pretrained(checkpoint).to(device)

In [3]:
def compile_augmentation_prompt(task, tokenizer):
    sentence = task["sentence"]
    entities = task["annotations"]

    entity_texts = [e["text"] for e in entities]
    entity_list = ", ".join(entity_texts)

    # Build chat template
    if not entities:
        chat = [
            {
                "role": "system",
                "content": (
                    "You are a helpful assistant that paraphrases sentences. "
                    "Paraphrase the following sentence in one single sentence."
                ),
            },
            {
                "role": "user",
                "content": f"Sentence: {sentence}",
            },
        ]
    else:
        chat = [
            {
                "role": "system",
                "content": (
                    f"You are a helpful assistant that paraphrases sentences. "
                    f"Paraphrase the following sentence in one single sentence while keeping these entities unchanged: {entity_list}. "
                ),
            },
            {
                "role": "user",
                "content": f"Sentence: {sentence}",
            },
        ]

    prompt = tokenizer.apply_chat_template(
        chat,
        tokenize=False,
        add_generation_prompt=True,
    )

    return prompt

In [11]:
# apply generative augmentation to certain proportion of the dataset
prop = 0.005
split_idx = int(len(data) * prop)
data_to_augment = data[:split_idx]

# empty list to store new augmented data in
augmented_dataset = []

# loop through all sentences for which augmentation should be applied
for task in data_to_augment:
    # compile the prompt and get the prompt ids
    prompt = compile_augmentation_prompt(task, tokenizer=tokenizer)
    prompt_ids = tokenizer(prompt, return_tensors="pt", truncation=True).to(device)
    # generate the output and decode back to actual text
    outputs = model.generate(**prompt_ids)
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # get only the answer text
    new_text = generated_text.split("assistant\n")[-1]
    # find the spans of the entities and append to list
    new_task = find_entity_span(task, new_text)
    augmented_dataset.append(new_task)

In [ ]:
prop = 0.05
split_idx = int(len(data)*prop)
data_to_augment = data[split_idx]

synonym_augmentations = []

for task in data_to_augment:
    new_task = augmentation_entity(task)
    synonym_augmentations.append(new_task)

In [ ]:
# export generative augmentations
with open("../01_data/augmented_annotations.json", "w") as f:
    json.dump(new_dataset, f)

# export the synonym augmentations